In [1]:
# Tools for math, tables, and file paths
import numpy as np
import pandas as pd
from pathlib import Path

# Your feature builder from Phase 2
from sitebench.features import SiteFeatures

# Find the repo root (the folder with .git) so file paths work wherever the notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

# Load the training and validation sites; test stays locked until Phase 5
train = pd.read_csv(ROOT / "data" / "processed" / "train.csv", low_memory=False)
val = pd.read_csv(ROOT / "data" / "processed" / "val.csv", low_memory=False)

# Show the TRIR at several percentiles of the training sites
for p in [50, 60, 75, 90, 95]:
    # The TRIR below which p% of training sites fall
    print(f"{p}th percentile: TRIR {np.percentile(train['trir'], p):.2f}")

50th percentile: TRIR 1.67
60th percentile: TRIR 3.10
75th percentile: TRIR 5.68
90th percentile: TRIR 11.28
95th percentile: TRIR 16.91


In [2]:
# Learn the cutoffs from training data only: the TRIR at the 50th and 90th percentiles
LOW_CUTOFF = np.percentile(train["trir"], 50)
HIGH_CUTOFF = np.percentile(train["trir"], 90)


def risk_level(trir):
    """Label each site low, medium, or high using the training cutoffs."""
    # np.select checks the conditions in order and uses the first one that's true;
    # anything that fails both gets the default, "high"
    return np.select(
        [trir < LOW_CUTOFF, trir < HIGH_CUTOFF],
        ["low", "medium"],
        default="high",
    )


# Add a risk_level column to both datasets
train["risk_level"] = risk_level(train["trir"])
val["risk_level"] = risk_level(val["trir"])

# Show what share of sites landed in each level (normalize=True gives fractions instead of counts)
print("Train:", train["risk_level"].value_counts(normalize=True).round(3).to_dict())
print("Val:  ", val["risk_level"].value_counts(normalize=True).round(3).to_dict())

Train: {'low': 0.5, 'medium': 0.4, 'high': 0.1}
Val:   {'low': 0.497, 'medium': 0.402, 'high': 0.101}


In [3]:
def gini(labels):
    """How mixed a group is: 0 = all one risk level, higher = more mixed."""
    # np.unique finds each distinct risk level and counts how many sites have it
    _, counts = np.unique(labels, return_counts=True)
    # Turn counts into shares, like 4 of 5 sites -> 0.8
    shares = counts / len(labels)
    # TODO: 1 minus the sum of the squared shares
    return (1 - np.sum(shares ** 2))


# The three groups you worked by hand
print(gini(["L", "L", "L", "L", "L", "M", "M", "M", "M", "H"]))  # the full toy set
print(gini(["M", "M", "M", "M", "H"]))                            # Question B's yes group
print(gini(["L", "L", "L", "L", "L"]))                            # Question B's no group

0.58
0.31999999999999984
0.0


In [4]:
def split_score(values, labels, threshold):
    """Weighted Gini after asking 'is this feature above the threshold?' Lower is better."""
    # Convert to NumPy arrays so we can filter them with True/False lists
    values = np.asarray(values)
    labels = np.asarray(labels)
    # True for every site that answers "yes" (above the threshold), False otherwise
    yes = values > threshold
    # Pick out each group's labels; ~yes flips True and False to get the "no" sites
    yes_labels = labels[yes]
    no_labels = labels[~yes]
    # Each group's share of all the sites, used as its weight
    yes_share = len(yes_labels) / len(labels)
    no_share = len(no_labels) / len(labels)
    # TODO: each group's share times its Gini, added together
    return yes_share * gini(yes_labels) + no_share * gini(no_labels)


# Toy data: the same 10 sites, now with hours per employee for each
hours = [1500, 1700, 1800, 1900, 2000, 2300, 2400, 2500, 2600, 2800]
levels = ["L", "L", "L", "L", "L", "M", "M", "M", "M", "H"]

print(split_score(hours, levels, 2200))
print(split_score(hours, levels, 1850))

0.15999999999999992
0.39999999999999997


In [5]:
def best_threshold(values, labels):
    """Try every midpoint between neighboring values; return the best cutoff and its score."""
    # np.unique sorts the distinct values, so neighbors sit next to each other
    unique_values = np.unique(values)
    # [:-1] is every value except the last, [1:] every value except the first,
    # so adding them pairs each value with its next neighbor; halving gives the midpoints
    candidates = (unique_values[:-1] + unique_values[1:]) / 2
    # No best cutoff yet; starting the best score at infinity means any real score beats it
    best_cut, best_score = None, np.inf
    for threshold in candidates:
        # TODO: score this cutoff using your split_score function
        score = split_score(values, labels, threshold)
        # If it beats the best so far, remember this cutoff and its score
        if score < best_score:
            best_cut, best_score = threshold, score
    return best_cut, best_score


print(best_threshold(hours, levels))

(np.float64(2150.0), np.float64(0.15999999999999992))


In [6]:
# Tables of features need pandas
import pandas as pd

# Toy data: 10 sites, two features each
toy_X = pd.DataFrame({
    "hours_per_employee": [1500, 1700, 1800, 1900, 2000, 2300, 2400, 2450, 2500, 2600],
    "is_mail_delivery":   [0,    0,    0,    0,    0,    0,    0,    1,    0,    0],
})
toy_levels = ["L", "L", "L", "L", "L", "M", "M", "H", "M", "M"]


def best_split(X, labels):
    """Try every feature and every cutoff; return the best feature, cutoff, and score."""
    # No best yet: no feature, no cutoff, and a score of infinity that anything beats
    best_feature, best_cut, best_score = None, None, np.inf
    # Repeat for each column name in the feature table
    for feature in X.columns:
        # TODO: find the best cutoff for this one feature, using best_threshold.
        # Its inputs are this feature's column of values, X[feature], and the labels
        cut, score = best_threshold(X[feature], labels)
        # If this feature's best beats the best so far, remember it
        if score < best_score:
            best_feature, best_cut, best_score = feature, cut, score
    return best_feature, best_cut, best_score


print(best_split(toy_X, toy_levels))

('hours_per_employee', np.float64(2150.0), np.float64(0.15999999999999992))


In [7]:
def majority(labels):
    """The most common risk level in a group: what a leaf answers."""
    # Count how many sites have each level, then pick the level with the biggest count
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]


def build_tree(X, labels, depth=0, max_depth=2):
    """Grow a tree: find the best question, split, then grow a smaller tree on each side."""
    labels = np.asarray(labels)
    # Stop and make a leaf if the group is already pure or the tree is as deep as allowed
    if gini(labels) == 0 or depth == max_depth:
        return majority(labels)
    # Find the best question for this group of sites
    feature, cut, score = best_split(X, labels)
    # If no question can split the group (every feature value identical), make a leaf
    if feature is None:
        return majority(labels)
    # True for each site that answers yes to the question
    yes = (X[feature] > cut).to_numpy()
    # A question node: the question, plus a whole subtree for each answer
    return {
        "question": f"{feature} > {cut}",
        "feature": feature,
        "cut": cut,
        # Grow a tree on only the yes sites, one level deeper
        "yes": build_tree(X[yes], labels[yes], depth + 1, max_depth),
        # TODO: same thing for the no sites (~yes flips True and False)
        "no": build_tree(X[~yes], labels[~yes], depth + 1, max_depth),
    }


def print_tree(node, indent=""):
    """Show the tree as indented if/else rules."""
    # A leaf is just a risk level, so print it and stop
    if not isinstance(node, dict):
        print(indent + "-> " + node)
        return
    # A question node: print the question, then each side's subtree, indented further
    print(indent + "if " + node["question"] + ":")
    print_tree(node["yes"], indent + "    ")
    print(indent + "else:")
    print_tree(node["no"], indent + "    ")


toy_tree = build_tree(toy_X, toy_levels, max_depth=2)
print_tree(toy_tree)

if hours_per_employee > 2150.0:
    if is_mail_delivery > 0.5:
        -> H
    else:
        -> M
else:
    -> L


In [9]:
# scikit-learn's decision tree, plus a helper that prints its rules as text
from sklearn.tree import DecisionTreeClassifier, export_text

# Same toy data, same depth limit, Gini as the scoring rule
sk_tree = DecisionTreeClassifier(criterion="gini", max_depth=2, random_state=0).fit(toy_X, toy_levels)

# Print its rules, labeled with your feature names
print(export_text(sk_tree, feature_names=list(toy_X.columns)))

|--- hours_per_employee <= 2150.00
|   |--- class: L
|--- hours_per_employee >  2150.00
|   |--- is_mail_delivery <= 0.50
|   |   |--- class: M
|   |--- is_mail_delivery >  0.50
|   |   |--- class: H



In [10]:
# Build the real feature tables, learning the encoding from training data only
features = SiteFeatures().fit(train)
X_train = features.transform(train)
X_val = features.transform(val)

# The targets: each site's risk level
y_train = train["risk_level"]
y_val = val["risk_level"]

# Try trees from 1 question deep up to 15
for depth in range(1, 16):
    # Train a tree with this depth limit; random_state makes ties break the same way every run
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    # .score returns accuracy: the share of sites predicted correctly
    train_acc = tree.score(X_train, y_train)
    val_acc = tree.score(X_val, y_val)
    # :>2 pads the depth so the columns line up; :.3f shows 3 decimals
    print(f"depth {depth:>2}: train {train_acc:.3f}   val {val_acc:.3f}")

depth  1: train 0.592   val 0.592
depth  2: train 0.617   val 0.616
depth  3: train 0.621   val 0.619
depth  4: train 0.627   val 0.625
depth  5: train 0.628   val 0.626
depth  6: train 0.633   val 0.629
depth  7: train 0.637   val 0.632
depth  8: train 0.641   val 0.636
depth  9: train 0.643   val 0.636
depth 10: train 0.649   val 0.640
depth 11: train 0.654   val 0.641
depth 12: train 0.659   val 0.642
depth 13: train 0.663   val 0.644
depth 14: train 0.668   val 0.643
depth 15: train 0.672   val 0.641


In [12]:
# Tools for the confusion matrix and recall
from sklearn.metrics import confusion_matrix, recall_score

# Train the depth-5 tree, your readability limit
tree5 = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_train, y_train)

# Predict a risk level for every validation site
pred5 = tree5.predict(X_val)

# Fix the order so the table reads low, medium, high
levels = ["low", "medium", "high"]

# Rows = true level, columns = predicted level; each cell counts sites
cm = confusion_matrix(y_val, pred5, labels=levels)
display(pd.DataFrame(cm, index=[f"true {l}" for l in levels], columns=[f"pred {l}" for l in levels]))

# Recall for each level: of the sites truly at that level, the share predicted at that level
for level, r in zip(levels, recall_score(y_val, pred5, labels=levels, average=None)):
    print(f"Recall for {level}: {r:.1%}")

,pred low,pred medium,pred high
true low,16312,12773,42
true medium,3449,20137,1
true high,2856,2798,254


Recall for low: 56.0%
Recall for medium: 85.4%
Recall for high: 4.3%


In [13]:
# More scoring tools: overall accuracy and precision
from sklearn.metrics import accuracy_score, precision_score

# "balanced" weights each level by how rare it is, so a high-risk site counts about 5x as much as a low one
tree5_w = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=0).fit(X_train, y_train)
pred5_w = tree5_w.predict(X_val)

# Confusion matrix for the weighted tree, same layout as before
cm_w = confusion_matrix(y_val, pred5_w, labels=levels)
display(pd.DataFrame(cm_w, index=[f"true {l}" for l in levels], columns=[f"pred {l}" for l in levels]))

# Compare both trees side by side on the three scores that matter
for name, pred in [("Unweighted", pred5), ("Balanced", pred5_w)]:
    acc = accuracy_score(y_val, pred)
    # [0] pulls the single number out, since we only asked about "high"
    rec_high = recall_score(y_val, pred, labels=["high"], average=None)[0]
    prec_high = precision_score(y_val, pred, labels=["high"], average=None, zero_division=0)[0]
    print(f"{name:>10}: accuracy {acc:.1%}   high recall {rec_high:.1%}   high precision {prec_high:.1%}")

,pred low,pred medium,pred high
true low,11203,11154,6770
true medium,3247,18169,2171
true high,1415,2128,2365


Unweighted: accuracy 62.6%   high recall 4.3%   high precision 85.5%
  Balanced: accuracy 54.1%   high recall 40.0%   high precision 20.9%


In [14]:
# The tree stores its levels alphabetically; find which column holds "high"
high_col = list(tree5.classes_).index("high")

# Each validation site's chance of being high risk:
# the share of high-risk training sites in the leaf it lands in
p_high = tree5.predict_proba(X_val)[:, high_col]

# True/False for each validation site: is it truly high risk?
truly_high = (y_val == "high").to_numpy()

for cutoff in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    # Flag a site as high if its chance is at or above the cutoff
    flagged = p_high >= cutoff
    # Recall: of the truly high sites, the share that got flagged
    recall = (flagged & truly_high).sum() / truly_high.sum()
    # Precision: of the flagged sites, the share truly high (max(..., 1) avoids dividing by zero)
    precision = (flagged & truly_high).sum() / max(flagged.sum(), 1)
    # :>6, adds a thousands comma and pads; :5.1% shows a percent with 1 decimal
    print(f"cutoff {cutoff:.2f}: flags {flagged.sum():>6,} sites ({flagged.mean():5.1%})   "
          f"recall {recall:5.1%}   precision {precision:5.1%}")

cutoff 0.10: flags 26,701 sites (45.5%)   recall 67.3%   precision 14.9%
cutoff 0.15: flags  8,541 sites (14.6%)   recall 29.8%   precision 20.6%
cutoff 0.20: flags  3,196 sites ( 5.5%)   recall 16.1%   precision 29.7%
cutoff 0.25: flags  1,010 sites ( 1.7%)   recall  8.3%   precision 48.3%
cutoff 0.30: flags    666 sites ( 1.1%)   recall  6.7%   precision 59.3%
cutoff 0.40: flags    484 sites ( 0.8%)   recall  5.6%   precision 67.8%
cutoff 0.50: flags    297 sites ( 0.5%)   recall  4.3%   precision 85.5%


In [15]:
# Try several pruning strengths; with no max_depth, each tree first grows as deep as it wants
for alpha in [0.0, 0.00001, 0.00005, 0.0001, 0.0005, 0.001]:
    # ccp_alpha is the pruning penalty per leaf
    pruned = DecisionTreeClassifier(ccp_alpha=alpha, random_state=0).fit(X_train, y_train)
    # get_depth and get_n_leaves show how big the tree is after pruning
    print(f"alpha {alpha:<8}: depth {pruned.get_depth():>2}, leaves {pruned.get_n_leaves():>6,}, "
          f"val accuracy {pruned.score(X_val, y_val):.3f}")

alpha 0.0     : depth 122, leaves 91,456, val accuracy 0.580
alpha 1e-05   : depth 68, leaves  2,563, val accuracy 0.646
alpha 5e-05   : depth 31, leaves    167, val accuracy 0.651
alpha 0.0001  : depth 19, leaves     86, val accuracy 0.647
alpha 0.0005  : depth 10, leaves     27, val accuracy 0.634
alpha 0.001   : depth  7, leaves     15, val accuracy 0.626


In [16]:
# Depth capped at 5 for readable rules; then prune at several strengths within that
for alpha in [0.0, 0.0001, 0.0005, 0.001, 0.002]:
    t = DecisionTreeClassifier(max_depth=5, ccp_alpha=alpha, random_state=0).fit(X_train, y_train)
    # Each validation site's chance of being high, then your rule: flag at 0.15 or above
    p = t.predict_proba(X_val)[:, list(t.classes_).index("high")]
    flagged = p >= 0.15
    # Recall and precision for high, same as before
    recall = (flagged & truly_high).sum() / truly_high.sum()
    precision = (flagged & truly_high).sum() / max(flagged.sum(), 1)
    print(f"alpha {alpha:<6}: leaves {t.get_n_leaves():>2}, val accuracy {t.score(X_val, y_val):.3f}, "
          f"high recall {recall:5.1%}, high precision {precision:5.1%}")

alpha 0.0   : leaves 32, val accuracy 0.626, high recall 29.8%, high precision 20.6%
alpha 0.0001: leaves 26, val accuracy 0.626, high recall 32.0%, high precision 20.2%
alpha 0.0005: leaves 17, val accuracy 0.626, high recall 33.3%, high precision 19.7%
alpha 0.001 : leaves 13, val accuracy 0.621, high recall 20.7%, high precision 24.6%
alpha 0.002 : leaves  8, val accuracy 0.619, high recall 19.2%, high precision 24.7%
